In [1]:
import pandas as pd

X = pd.read_csv("../data/processed/X_features.csv")
y = pd.read_csv("../data/processed/y_target.csv").squeeze()  # squeeze turns single-column df into a Series

print(X.shape, y.shape)

(2392, 16) (2392,)


In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (1913, 16) Test: (479, 16)


In [3]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

         0.0       0.12      0.33      0.18        21
         1.0       0.40      0.31      0.35        54
         2.0       0.51      0.54      0.52        78
         3.0       0.44      0.51      0.47        83
         4.0       0.95      0.78      0.86       243

    accuracy                           0.62       479
   macro avg       0.48      0.49      0.47       479
weighted avg       0.69      0.62      0.65       479



c:\EduAlert\Student-Early-Warning-System\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [4]:
from sklearn.tree import DecisionTreeClassifier

dtree = DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42)
dtree.fit(X_train, y_train)

y_pred_dt = dtree.predict(X_test)
print(classification_report(y_test, y_pred_dt))

              precision    recall  f1-score   support

         0.0       0.18      0.19      0.19        21
         1.0       0.32      0.46      0.38        54
         2.0       0.37      0.36      0.37        78
         3.0       0.37      0.49      0.42        83
         4.0       0.93      0.74      0.82       243

    accuracy                           0.58       479
   macro avg       0.43      0.45      0.43       479
weighted avg       0.64      0.58      0.60       479



In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

              precision    recall  f1-score   support

         0.0       0.25      0.19      0.22        21
         1.0       0.53      0.50      0.51        54
         2.0       0.52      0.56      0.54        78
         3.0       0.42      0.57      0.48        83
         4.0       0.93      0.83      0.88       243

    accuracy                           0.67       479
   macro avg       0.53      0.53      0.53       479
weighted avg       0.70      0.67      0.68       479



In [10]:
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.utils import to_categorical # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Dense, Dropout # type: ignore

# Scale features - neural networks are sensitive to feature scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# One-hot encode target (5 classes: 0-4)
y_train_cat = to_categorical(y_train, num_classes=5)
y_test_cat = to_categorical(y_test, num_classes=5)

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(5, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train_scaled, y_train_cat,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    verbose=1
)

Epoch 1/50


c:\EduAlert\Student-Early-Warning-System\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3516 - loss: 1.5378 - val_accuracy: 0.5039 - val_loss: 1.2276
Epoch 2/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5353 - loss: 1.1808 - val_accuracy: 0.5326 - val_loss: 1.0574
Epoch 3/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5725 - loss: 1.0823 - val_accuracy: 0.5901 - val_loss: 0.9923
Epoch 4/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5876 - loss: 1.0403 - val_accuracy: 0.6319 - val_loss: 0.9658
Epoch 5/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6092 - loss: 0.9956 - val_accuracy: 0.6527 - val_loss: 0.9449
Epoch 6/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6307 - loss: 0.9787 - val_accuracy: 0.6815 - val_loss: 0.9286
Epoch 7/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6170 - loss: 0.9612 - val_accuracy: 0.6815 - val_loss: 0.9178
Epoch 8/50
48/48 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6621 - loss: 0.9285 - val_accuracy: 0.6945 - val_loss: 0.9006
Epo

In [8]:
import numpy as np

y_pred_nn_probs = model.predict(X_test_scaled)
y_pred_nn = np.argmax(y_pred_nn_probs, axis=1)

print(classification_report(y_test, y_pred_nn))

 1/15 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
              precision    recall  f1-score   support

         0.0       0.50      0.10      0.16        21
         1.0       0.59      0.65      0.62        54
         2.0       0.57      0.59      0.58        78
         3.0       0.52      0.52      0.52        83
         4.0       0.88      0.92      0.90       243

    accuracy                           0.73       479
   macro avg       0.61      0.55      0.56       479
weighted avg       0.72      0.73      0.72       479



In [9]:
import joblib

joblib.dump(log_reg, '../models/logistic_regression.pkl')
joblib.dump(dtree, '../models/decision_tree.pkl')
joblib.dump(rf, '../models/random_forest.pkl')
joblib.dump(scaler, '../models/scaler.pkl')  # needed to preprocess new data for the NN later
model.save('../models/neural_network.keras')